# Feature Engineering — IEEE-CIS Fraud Detection

**Objetivo de este notebook:** crear nuevas variables a partir de los hallazgos del análisis exploratorio, para que los modelos puedan detectar mejor el fraude.

En el notebook 01 comprobamos que ninguna variable explica el fraude por sí sola (la correlación máxima con el target es de 0,15), pero que hay señales claras en la ausencia de datos, en el importe, en algunas categorías y en las interacciones entre variables. En este notebook convertimos esas señales en variables:
- **Separación temporal** entre entrenamiento y validación, antes de crear cualquier variable que aprenda de los datos.
- **Variables por reglas fijas:** indicadores de ausencia de datos, familias de sistema operativo y navegador, marca del dispositivo, variables de tiempo y transformaciones del importe.
- **Variables agregadas:** resúmenes del comportamiento de cada tarjeta, calculados sin usar información del periodo de validación.
- **Comprobación:** relación de cada variable nueva con el fraude.

Guardamos los conjuntos de entrenamiento, validación y test con las nuevas variables. El tratamiento de nulos, el encoding y el escalado dependen de cada modelo, por lo que se aplicarán dentro de los pipelines del notebook de modelado.

In [14]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [15]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid", palette="deep", rc={"figure.figsize": (10, 5)})

sorted(p.name for p in PROCESSED_DIR .glob("*.parquet"))

['test_clean.parquet', 'train_clean.parquet']

In [16]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")

print(train.shape)
print(test.shape)

(590537, 434)
(506691, 433)


## 1. Split

In [17]:
corte = train["TransactionDT"].quantile(0.8)
train["split"] = np.where(train["TransactionDT"] <= corte, "train", "valid")
test["split"] = "test"

df = pd.concat([train, test], ignore_index=True)
del train, test

df["split"].value_counts()

split
test     506691
train    472429
valid    118108
Name: count, dtype: int64

Split temporal, hay que ordenar por fecha, unimos todo para asi no hacer cada accion tres veces

## 2.Nulos

Descubrimos que cada modelo decidirá que hacer con los nulos, en este caso haremos solo 3 que vimos que si influian

In [20]:
df['has_identity'] = df['id_12'].notna()
df['has_addr']= df['addr1'].notna()
df['has_r_email'] = df['R_emaildomain'].notna()



**Sistema operativo**

In [19]:
df["os_family"] = df["id_30"].str.split().str[0]
df["os_family"] = df["id_30"].str.split().str[0]

In [23]:
print(df["os_family"].unique())
df[df["split"] == "train"].groupby("os_family", dropna=False)["isFraud"].mean()

[nan 'android' 'ios' 'mac' 'windows' 'linux' 'func' 'other']


os_family
android   0.086
func      0.000
ios       0.061
linux     0.082
mac       0.018
other     0.333
windows   0.035
NaN       0.034
Name: isFraud, dtype: float64

## Familia del sistema operativo

`id_30` contiene la versión exacta del sistema operativo (`windows 10`, `ios 11.4.1`, `mac os x 10_14_6`…). Hay decenas de versiones y muchas tienen muy pocas transacciones, así que el modelo no puede aprender un patrón fiable de cada una. Agrupamos las versiones en familias quedándonos con la primera palabra, de modo que cada grupo reúne suficientes filas.

Calculamos la tasa de fraude por familia **solo con las filas de entrenamiento**, para no usar información de validación ni de test:

- **Android (8,6%), Linux (8,2%) e iOS (6,1%)** superan con claridad la tasa general del 3,5%.
- **Windows (3,5%)** está en la media y **Mac (1,8%)** queda por debajo.
- **`other` (33,3%) y `func` (0%)** muestran valores extremos, pero se calculan sobre muy pocas filas (XX y XX), por lo que no son fiables. `func` es un valor sin documentar en el dataset original.

**Conclusión:** la familia del sistema